# Understanding Data Formats for MPLAB ML

Learn the basics of preparing data for MPLAB ML:
- Sequential vs Wide format
- Loading and inspecting data
- Finding common data issues
- Basic visualization techniques

**What you'll learn:**
- ✅ The format MPLAB ML expects
- ✅ How to check your data quality
- ✅ Simple ways to visualize sensor data
- ✅ How to spot mislabeled or missing data

---
## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for better-looking plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 4)

print("✓ Libraries imported")

---
## 2. Data Formats: Sequential vs Wide

### Sequential Format (MPLAB ML Standard)

**Structure:** One reading per row, with separate columns for each sensor

```
Time        AccelX    AccelY    AccelZ    Label
0.000       0.12      -0.05     9.81      standing
0.010       0.13      -0.04     9.80      standing
0.020       0.15      -0.06     9.82      standing
```

**Key characteristics:**
- ✅ Time increases down the rows
- ✅ Each sensor has its own column
- ✅ One label column (optional)
- ✅ MPLAB ML can process this directly

---

### Wide Format (Needs Conversion)

**Structure:** Multiple readings in a single row

```
Sample    AccelX_1  AccelX_2  AccelX_3  AccelY_1  AccelY_2  AccelY_3  ...
1         0.12      0.13      0.15      -0.05     -0.04     -0.06     ...
```

**Key characteristics:**
- ❌ Not compatible with MPLAB ML
- ❌ Needs conversion to sequential format
- ❌ Common in some data logging systems

**This notebook focuses on sequential format - the standard for MPLAB ML.**

---
## 3. Load Sample Data

Let's create some example sensor data to explore:

In [ ]:
# Create sample sequential data (simulated 3-axis accelerometer)
np.random.seed(42)
n_samples = 1000

# Simulate three different activities
time = np.arange(n_samples) * 0.01  # 100Hz sampling (0.01s per sample)

# Activity 1: Standing (low variance)
standing_x = np.random.normal(0.1, 0.05, 300)
standing_y = np.random.normal(-0.05, 0.05, 300)
standing_z = np.random.normal(9.8, 0.1, 300)

# Activity 2: Walking (medium variance with pattern)
walking_x = np.sin(np.arange(400) * 0.1) * 0.5 + np.random.normal(0, 0.1, 400)
walking_y = np.cos(np.arange(400) * 0.1) * 0.3 + np.random.normal(0, 0.1, 400)
walking_z = 9.8 + np.random.normal(0, 0.3, 400)

# Activity 3: Running (high variance)
running_x = np.sin(np.arange(300) * 0.3) * 1.5 + np.random.normal(0, 0.3, 300)
running_y = np.cos(np.arange(300) * 0.3) * 1.0 + np.random.normal(0, 0.3, 300)
running_z = 9.8 + np.random.normal(0, 0.8, 300)

# Combine all activities
accel_x = np.concatenate([standing_x, walking_x, running_x])
accel_y = np.concatenate([standing_y, walking_y, running_y])
accel_z = np.concatenate([standing_z, walking_z, running_z])
labels = ['standing']*300 + ['walking']*400 + ['running']*300

# Create DataFrame in sequential format
df = pd.DataFrame({
    'Time': time,
    'AccelX': accel_x,
    'AccelY': accel_y,
    'AccelZ': accel_z,
    'Label': labels
})

# Add some realistic issues
df.loc[150, 'AccelX'] = np.nan  # Missing value
df.loc[500, 'Label'] = 'standng'  # Typo in label
df.loc[850, 'AccelZ'] = 50.0  # Outlier

print(f"✓ Created {len(df)} samples of sequential sensor data")
print(f"  Sampling rate: 100 Hz")
print(f"  Duration: {df['Time'].max():.2f} seconds")

---
## 4. Basic Data Inspection

Always start by looking at your data!

In [ ]:
# First 10 rows
print("First 10 rows:")
display(df.head(10))

In [ ]:
# Data info
print("Dataset Information:")
print(df.info())

In [ ]:
# Basic statistics
print("\nStatistics for sensor channels:")
display(df[['AccelX', 'AccelY', 'AccelZ']].describe())

In [ ]:
# Check label distribution
print("\nLabel distribution:")
print(df['Label'].value_counts())

---
## 5. Finding Data Issues

### 5.1 Check for Missing Values

In [ ]:
# Count missing values per column
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0])

# Percentage of missing data
missing_pct = (missing / len(df)) * 100
print("\nPercentage missing:")
print(missing_pct[missing_pct > 0])

if missing.sum() > 0:
    print("\n⚠️  Found missing values! Consider:")
    print("   - Removing rows with missing data")
    print("   - Interpolating missing values")
    print("   - Investigating why data is missing")

### 5.2 Check for Outliers

In [ ]:
# Simple outlier detection using z-score
def find_outliers(df, columns, threshold=3):
    """Find outliers using z-score method."""
    outliers = pd.DataFrame()
    
    for col in columns:
        z_scores = np.abs((df[col] - df[col].mean()) / df[col].std())
        outlier_mask = z_scores > threshold
        col_outliers = df[outlier_mask].copy()
        col_outliers['outlier_column'] = col
        col_outliers['z_score'] = z_scores[outlier_mask]
        outliers = pd.concat([outliers, col_outliers])
    
    return outliers

# Find outliers in sensor data
sensor_cols = ['AccelX', 'AccelY', 'AccelZ']
outliers = find_outliers(df, sensor_cols, threshold=3)

if len(outliers) > 0:
    print(f"⚠️  Found {len(outliers)} potential outliers:\n")
    display(outliers[['Time', 'AccelX', 'AccelY', 'AccelZ', 'outlier_column', 'z_score']])
else:
    print("✓ No significant outliers detected")

### 5.3 Check for Label Issues

In [ ]:
# Check for unexpected or mislabeled values
print("Unique labels in dataset:")
unique_labels = df['Label'].unique()
print(unique_labels)

# Expected labels
expected_labels = ['standing', 'walking', 'running']

# Find any unexpected labels
unexpected = set(unique_labels) - set(expected_labels)
if unexpected:
    print(f"\n⚠️  Found unexpected labels: {unexpected}")
    print("   These may be typos or mislabeled data")
    
    # Show examples
    for label in unexpected:
        count = (df['Label'] == label).sum()
        print(f"   '{label}': {count} occurrences")
else:
    print("\n✓ All labels match expected values")

---
## 6. Data Visualization

### 6.1 Time Series Plot

In [ ]:
# Plot all three axes
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

# Color code by label
colors = {'standing': 'green', 'walking': 'orange', 'running': 'red'}

for i, col in enumerate(['AccelX', 'AccelY', 'AccelZ']):
    for label in df['Label'].unique():
        if label in colors:  # Only plot valid labels
            mask = df['Label'] == label
            axes[i].scatter(df[mask]['Time'], df[mask][col], 
                          c=colors[label], alpha=0.5, s=1, label=label)
    
    axes[i].set_ylabel(col)
    axes[i].grid(True, alpha=0.3)
    if i == 0:
        axes[i].legend()

axes[2].set_xlabel('Time (seconds)')
plt.suptitle('Accelerometer Data - All Axes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 Notice:")
print("   - Standing (green): Low variance, stable signal")
print("   - Walking (orange): Medium variance, periodic pattern")
print("   - Running (red): High variance, larger oscillations")

### 6.2 Distribution Plots

In [ ]:
# Distribution of each sensor by activity
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, col in enumerate(['AccelX', 'AccelY', 'AccelZ']):
    for label in ['standing', 'walking', 'running']:
        data = df[df['Label'] == label][col].dropna()
        axes[i].hist(data, alpha=0.5, bins=30, label=label)
    
    axes[i].set_xlabel(f'{col} (m/s²)')
    axes[i].set_ylabel('Count')
    axes[i].set_title(f'{col} Distribution')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 Distributions show:")
print("   - Overlap between activities (classification challenge)")
print("   - Range of values for each sensor")
print("   - Relative variance between activities")

### 6.3 Box Plots by Label

In [ ]:
# Box plot to compare sensor values across activities
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, col in enumerate(['AccelX', 'AccelY', 'AccelZ']):
    # Only plot valid labels
    valid_data = df[df['Label'].isin(['standing', 'walking', 'running'])]
    valid_data.boxplot(column=col, by='Label', ax=axes[i])
    axes[i].set_xlabel('Activity')
    axes[i].set_ylabel(f'{col} (m/s²)')
    axes[i].set_title(f'{col} by Activity')
    axes[i].get_figure().suptitle('')  # Remove default title

plt.tight_layout()
plt.show()

print("💡 Box plots reveal:")
print("   - Median values for each activity")
print("   - Variance (box height)")
print("   - Outliers (dots beyond whiskers)")

---
## 7. Data Quality Checklist

Before uploading to MPLAB ML, verify:

### ✅ Format Requirements
- [ ] Data is in **sequential format** (one reading per row)
- [ ] Time column exists and increases monotonically
- [ ] Each sensor has its own column
- [ ] Label column present (if using labeled data)

### ✅ Data Quality
- [ ] No missing values (or handled appropriately)
- [ ] No obvious outliers (or documented)
- [ ] Labels are spelled correctly and consistent
- [ ] Data range is reasonable for your sensors

### ✅ Sufficient Data
- [ ] Enough samples per label (typically 100+ per class)
- [ ] Balanced across labels (or intentionally imbalanced)
- [ ] Representative of real-world conditions

---
## 8. Quick Fix Examples

### Remove Missing Values

In [ ]:
# Option 1: Drop rows with any missing values
df_clean = df.dropna()
print(f"Removed {len(df) - len(df_clean)} rows with missing data")

# Option 2: Forward fill (use previous value)
df_filled = df.fillna(method='ffill')
print(f"Filled {df.isnull().sum().sum()} missing values using forward fill")

### Fix Label Typos

In [ ]:
# Fix the typo we introduced
df_fixed = df.copy()
df_fixed.loc[df_fixed['Label'] == 'standng', 'Label'] = 'standing'

print("Labels after fixing:")
print(df_fixed['Label'].value_counts())

### Remove Outliers

In [ ]:
# Remove extreme outliers (z-score > 4)
def remove_outliers(df, columns, threshold=4):
    df_clean = df.copy()
    for col in columns:
        z_scores = np.abs((df_clean[col] - df_clean[col].mean()) / df_clean[col].std())
        df_clean = df_clean[z_scores < threshold]
    return df_clean

df_no_outliers = remove_outliers(df_fixed, ['AccelX', 'AccelY', 'AccelZ'])
print(f"Removed {len(df_fixed) - len(df_no_outliers)} outlier samples")

---
## 9. Export Clean Data

In [ ]:
# Save cleaned data
output_file = 'clean_sensor_data.csv'
df_no_outliers.to_csv(output_file, index=False)

print(f"✓ Saved clean data to: {output_file}")
print(f"  Original samples: {len(df)}")
print(f"  Clean samples: {len(df_no_outliers)}")
print(f"  Removed: {len(df) - len(df_no_outliers)} ({((len(df) - len(df_no_outliers))/len(df)*100):.1f}%)")
print("\n✓ Data is ready for MPLAB ML!")

---
## Summary

### Key Takeaways:

**Data Format:**
- ✅ MPLAB ML requires **sequential format**
- ✅ One reading per row, one column per sensor
- ✅ Time increases down the rows

**Data Quality Checks:**
1. Missing values → Remove or interpolate
2. Outliers → Investigate and remove if needed
3. Label typos → Fix before upload
4. Data range → Verify it's realistic

**Visualization:**
- Time series plots show patterns and anomalies
- Distributions reveal data characteristics
- Box plots compare activities

**Next Steps:**
- [Labeling Data](../labeling-data/) - Add or refine labels
- [Creating Manifests](../labeling-data/creating-manifests.ipynb) - Build .dclproj files
- [Complete Workflows](../complete-workflows/) - End-to-end examples

---

**Pro Tips:**
- 💡 Always visualize your data before uploading
- 💡 Check for label consistency across your team
- 💡 Document any data cleaning decisions
- 💡 Keep original data - clean in a copy